In [10]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

dataset_id = "wikimedia/wikipedia"
dataset_config = "20231101.zh-classical"
# dataset_config = "20231101.ab"

df = pd.read_parquet(f"datasets/{dataset_id}/{dataset_config}/train-00000-of-00001.parquet")
table = pa.Table.from_pandas(df)

# Output file path
out_file = f"datasets_compress/{dataset_id}/{dataset_config}-test.parquet"

# Compression type (e.g., 'snappy', 'gzip', 'brotli', 'zstd', or None)
PARQUET_COMPRESSION_TYPE = "snappy"

# Write to Parquet
pq.write_table(table, out_file, compression=PARQUET_COMPRESSION_TYPE, row_group_size=1000)
print(df.head())

   id                                                url title  \
0   5  https://zh-classical.wikipedia.org/wiki/%E8%A5...  西楚霸王   
1  10  https://zh-classical.wikipedia.org/wiki/%E9%BB...    黃帝   
2  50  https://zh-classical.wikipedia.org/wiki/%E5%B8...    帝舜   
3  51  https://zh-classical.wikipedia.org/wiki/%E5%B8...    帝堯   
4  54  https://zh-classical.wikipedia.org/wiki/%E9%A1...   顧愷之   

                                                text  
0  西楚霸王項籍，字羽，下相人也。初起，年二十四。其季父梁。嘗殺人，與籍避仇吳中。秦始皇帝東遊會...  
1  黃帝者，少典之子，姓公孫，名曰軒轅。生而神靈，弱而能言，幼而徇齊，長而敦敏，成而聰明。\n\...  
2  虞舜者，名曰重華。重華父曰瞽叟，瞽叟父曰橋牛，橋牛父曰句望，句望父曰敬康，敬康父曰窮蟬，窮蟬...  
3  帝堯，五帝也，陶唐氏，諱放勛，堯乃其諡。然《尚書集傳》曰：放勳意至功四方、不爲名。以爲集傳謂...  
4  顧愷之字長康，晉晉陵無錫人也。父悅之，尚書左丞。愷之博學有才氣，嘗為《箏賦》，成，謂人曰：「...  


In [11]:
compress_df = df.drop(columns=["title"])

In [12]:
print(compress_df.head())

   id                                                url  \
0   5  https://zh-classical.wikipedia.org/wiki/%E8%A5...   
1  10  https://zh-classical.wikipedia.org/wiki/%E9%BB...   
2  50  https://zh-classical.wikipedia.org/wiki/%E5%B8...   
3  51  https://zh-classical.wikipedia.org/wiki/%E5%B8...   
4  54  https://zh-classical.wikipedia.org/wiki/%E9%A1...   

                                                text  
0  西楚霸王項籍，字羽，下相人也。初起，年二十四。其季父梁。嘗殺人，與籍避仇吳中。秦始皇帝東遊會...  
1  黃帝者，少典之子，姓公孫，名曰軒轅。生而神靈，弱而能言，幼而徇齊，長而敦敏，成而聰明。\n\...  
2  虞舜者，名曰重華。重華父曰瞽叟，瞽叟父曰橋牛，橋牛父曰句望，句望父曰敬康，敬康父曰窮蟬，窮蟬...  
3  帝堯，五帝也，陶唐氏，諱放勛，堯乃其諡。然《尚書集傳》曰：放勳意至功四方、不爲名。以爲集傳謂...  
4  顧愷之字長康，晉晉陵無錫人也。父悅之，尚書左丞。愷之博學有才氣，嘗為《箏賦》，成，謂人曰：「...  


In [5]:
compress_df["combine_title_text"] = compress_df["title"] + compress_df["text"]
compress_df["title_len"] = compress_df["title"].str.len()
compress_df = compress_df.drop(columns=["title", "text"])

In [6]:
print(compress_df.head())

     id                                 combine_title_text  title_len
0   807  Аԥсуа бызшәаА́ԥсуа бызшәа́, аԥсшәа (, ) — Аԥсн...         12
1  1040  АҟәаА́ҟәа () — Аԥсны Аҳәынҭқарра  аҳҭнықалақь....          4
2  1044  Аԥсуа алфавит\n\nАҭоурых\nИахьа 142 шықәса ҵит...         13
3  1046  ГаграГа́гра (, ) — Аԥсны ақалақь. Ишьҭоуп Гaгр...          5
4  1053  Аԥсны АҳәынҭқарраАԥсны Аҳәынҭқарра () — ҳәҭакл...         17


In [15]:
compress_df.to_parquet(f"datasets_compress/{dataset_id}/{dataset_config}-2.parquet", index=False)

In [3]:
import pandas as pd
dataset_id = "wikimedia/wikipedia"
config_name = "20231101.zh-classical"
compress_df = pd.read_parquet(f"datasets_parquet/{dataset_id}/{config_name}.parquet")
print(compress_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12708 entries, 0 to 12707
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      12708 non-null  object
 1   url     12708 non-null  object
 2   title   12708 non-null  object
 3   text    12708 non-null  object
dtypes: object(4)
memory usage: 397.3+ KB
None


In [17]:
import os

# get the size of the dataset
def get_size(start_path = '.'):
    # check is it a file or directory
    if os.path.isfile(start_path):
        return os.path.getsize(start_path)
    total = 0
    for dirpath, dirnames, filenames in os.walk(start_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total


size = get_size(f"datasets/{dataset_id}/{dataset_config}")
test_size = get_size(f"datasets_compress/{dataset_id}/{dataset_config}-test.parquet")
compression_size = get_size(f"datasets_compress/{dataset_id}/{dataset_config}-2.parquet")
print(f"Size of the dataset: {size / (1024):.2f} kB")
print(f"Size of the test dataset: {test_size / (1024):.2f} kB")
print(f"Size of the compression: {compression_size / (1024):.2f} kB")
print(f"Compression ratio: {((size-compression_size) / size) * 100:.2f} %")
print(f"Test Compression ratio: {((test_size-compression_size) / test_size) * 100:.2f} %")


Size of the dataset: 9861.40 kB
Size of the test dataset: 9838.62 kB
Size of the compression: 9656.43 kB
Compression ratio: 2.08 %
Test Compression ratio: 1.85 %


In [18]:
from urllib.parse import quote, unquote
# compress_df["url"] = compress_df["title"].apply(lambda x: "https://zh-classical.wikipedia.org/wiki/" + quote(x))
# compress_df["title"] = compress_df.apply(
#     lambda r: r["combine_title_text"][: r["title_len"]], axis=1, result_type="reduce"
# )
# compress_df["text"] = compress_df.apply(
#     lambda r: r["combine_title_text"][r["title_len"] :], axis=1, result_type="reduce"
# )
# compress_df["url"] = compress_df["title"].apply(lambda x: "https://ab.wikipedia.org/wiki/" + quote(x))
# compress_df = compress_df.drop(columns=["combine_title_text", "title_len"])

prefix =  "https://zh-classical.wikipedia.org/wiki/"
compress_df["title"] = compress_df["url"].apply(lambda x: unquote(x.replace(prefix, "")))


In [19]:
print(compress_df.head())

   id                                                url  \
0   5  https://zh-classical.wikipedia.org/wiki/%E8%A5...   
1  10  https://zh-classical.wikipedia.org/wiki/%E9%BB...   
2  50  https://zh-classical.wikipedia.org/wiki/%E5%B8...   
3  51  https://zh-classical.wikipedia.org/wiki/%E5%B8...   
4  54  https://zh-classical.wikipedia.org/wiki/%E9%A1...   

                                                text title  
0  西楚霸王項籍，字羽，下相人也。初起，年二十四。其季父梁。嘗殺人，與籍避仇吳中。秦始皇帝東遊會...  西楚霸王  
1  黃帝者，少典之子，姓公孫，名曰軒轅。生而神靈，弱而能言，幼而徇齊，長而敦敏，成而聰明。\n\...    黃帝  
2  虞舜者，名曰重華。重華父曰瞽叟，瞽叟父曰橋牛，橋牛父曰句望，句望父曰敬康，敬康父曰窮蟬，窮蟬...    帝舜  
3  帝堯，五帝也，陶唐氏，諱放勛，堯乃其諡。然《尚書集傳》曰：放勳意至功四方、不爲名。以爲集傳謂...    帝堯  
4  顧愷之字長康，晉晉陵無錫人也。父悅之，尚書左丞。愷之博學有才氣，嘗為《箏賦》，成，謂人曰：「...   顧愷之  


In [20]:
def compare_dataframes(df1, df2):
    # Align columns by name
    df1_sorted = df1.sort_index(axis=1)
    df2_sorted = df2[df1_sorted.columns]  # reorder df2 to match df1

    # Check shape first
    if df1_sorted.shape != df2_sorted.shape:
        print("DataFrames have different shapes:", df1_sorted.shape, df2_sorted.shape)
        return

    # Create a boolean DataFrame of element-wise comparisons
    diff = df1_sorted != df2_sorted

    if not diff.any().any():
        print("DataFrames are equal")
    else:
        print("Differences found at these locations:")
        differing_cells = diff.stack()[diff.stack()]  # True where differences exist
        for (idx, col) in differing_cells.index:
            print(f"- Row {idx}, Column '{col}': df1 = {df1_sorted.at[idx, col]!r}, df2 = {df2_sorted.at[idx, col]!r}")


compare_dataframes(df, compress_df)

DataFrames are equal


In [27]:
import pyarrow.parquet as pq

# Load the file
parquet_file = pq.ParquetFile(f"datasets/{dataset_id}/{dataset_config}/train-00000-of-00001.parquet")

# Get compression for each column in the first row group
for i, col in enumerate(parquet_file.schema.names):
    enc = parquet_file.metadata.row_group(0).column(i).compression
    print(parquet_file.metadata.row_group(0).column(i))
    break
    # print(f"Column '{col}' uses compression: {enc}")

  file_offset: 5778
  file_path: 
  physical_type: BYTE_ARRAY
  num_values: 1000
  path_in_schema: id
  is_stats_set: True
  statistics:
      has_min_max: True
      min: 1040
      max: 807
      null_count: 0
      distinct_count: None
      num_values: 1000
      physical_type: BYTE_ARRAY
      logical_type: String
      converted_type (legacy): UTF8
  compression: SNAPPY
  encodings: ('RLE_DICTIONARY', 'PLAIN', 'RLE')
  has_dictionary_page: True
  dictionary_page_offset: 4
  data_page_offset: 4478
  total_compressed_size: 5774
  total_uncompressed_size: 9311


In [28]:

parquet_file = pq.ParquetFile(f"datasets_compress/{dataset_id}/{dataset_config}-test.parquet")

# Get compression for each column in the first row group
for i, col in enumerate(parquet_file.schema.names):
    enc = parquet_file.metadata.row_group(0).column(i).compression
    print(parquet_file.metadata.row_group(0).column(i))
    break
    # print(f"Column '{col}' uses compression: {enc}")

  file_offset: 0
  file_path: 
  physical_type: BYTE_ARRAY
  num_values: 1000
  path_in_schema: id
  is_stats_set: True
  statistics:
      has_min_max: True
      min: 1040
      max: 807
      null_count: 0
      distinct_count: None
      num_values: 1000
      physical_type: BYTE_ARRAY
      logical_type: String
      converted_type (legacy): UTF8
  compression: SNAPPY
  encodings: ('PLAIN', 'RLE', 'RLE_DICTIONARY')
  has_dictionary_page: True
  dictionary_page_offset: 4
  data_page_offset: 4478
  total_compressed_size: 5774
  total_uncompressed_size: 9311


In [9]:
import json
compression_methods = ["snappy", "gzip", "brotli", "lz4", "zstd"]
dataset_id = "wikimedia/wikipedia"
config_dict = json.load(open("config.json", "r"))
dataset_config = config_dict[dataset_id]
total_size = {
    "snappy": 0,
    "gzip": 0,
    "brotli": 0,
    "lz4": 0,
    "zstd": 0,
}
total_compress_size_1 = {
    "snappy": 0,
    "gzip": 0,
    "brotli": 0,
    "lz4": 0,
    "zstd": 0,
}
total_compress_size_2 = {
    "snappy": 0,
    "gzip": 0,
    "brotli": 0,
    "lz4": 0,
    "zstd": 0,
}
for config in config_dict[dataset_id]["configs"]:
    for compression_method in compression_methods:
        total_size[f"{compression_method}"] += config[f"size_{compression_method}"]
        total_compress_size_1[f"{compression_method}"] += config["compress"][0][f"size_compression_{compression_method}"]
        total_compress_size_2[f"{compression_method}"] += config["compress"][1][f"size_compression_{compression_method}"]
        

for compression_method in compression_methods:
    print(f"Total size of the dataset ({compression_method}): {total_size[f"{compression_method}"] / (1024):.2f} kB")
    print(f"Total size of the compression_1 ({compression_method}): {total_compress_size_1[f"{compression_method}"] / (1024):.2f} kB")
    print(f"Total compression_1 ratio ({compression_method}): {((total_size[f"{compression_method}"]-total_compress_size_1[f"{compression_method}"]) / total_size[f"{compression_method}"]) * 100:.2f} %")
    print(f"Total size of the compression_2 ({compression_method}): {total_compress_size_2[f"{compression_method}"] / (1024):.2f} kB")
    print(f"Total compression_2 ratio ({compression_method}): {((total_size[f"{compression_method}"]-total_compress_size_2[f"{compression_method}"]) / total_size[f"{compression_method}"]) * 100:.2f} %")


KeyError: 'size_compression_gzip'